# Construção do universo líquido por janela móvel

Nesta etapa, utilizamos as matrizes criadas a partir da COTAHIST da B3 para selecionar, em cada janela de formação, os ativos com dados suficientes e maior liquidez. seleciona os ativos válidos em cada período histórico com base na disponibilidade de dados e no volume financeiro médio negociado.

In [1]:
from pathlib import Path

import pandas as pd


PASTA_PROJETO = Path.cwd().parent
PASTA_DADOS_TRATADOS = PASTA_PROJETO / "dados_tratados"

precos = pd.read_csv(
    PASTA_DADOS_TRATADOS / "precos_b3_2010_2025.csv",
    index_col=0,
    parse_dates=True
)

volumes_financeiros = pd.read_csv(
    PASTA_DADOS_TRATADOS / "volumes_financeiros_b3_2010_2025.csv",
    index_col=0,
    parse_dates=True
)

negocios = pd.read_csv(
    PASTA_DADOS_TRATADOS / "negocios_b3_2010_2025.csv",
    index_col=0,
    parse_dates=True
)

print(f"Matriz de preços: {precos.shape[0]} datas e {precos.shape[1]} ativos")
print(f"Matriz de volume financeiro: {volumes_financeiros.shape[0]} datas e {volumes_financeiros.shape[1]} ativos")
print(f"Matriz de negócios: {negocios.shape[0]} datas e {negocios.shape[1]} ativos")

Matriz de preços: 3967 datas e 2051 ativos
Matriz de volume financeiro: 3967 datas e 2051 ativos
Matriz de negócios: 3967 datas e 2051 ativos


### 1. Definição dos critérios de liquidez

Nesta etapa, definimos os critérios que serão usados para selecionar os ativos líquidos em cada janela de formação. A ideia é manter apenas ações com dados suficientes, negociação recorrente e maior volume financeiro médio no período analisado.

Esses critérios são importantes porque o projeto não utilizará uma carteira fixa de ações. Em vez disso, o universo de ativos será construído dinamicamente em cada janela móvel, reduzindo o risco de viés de sobrevivência e evitando ativos com pouca negociação.

In [2]:
JANELA_FORMACAO_MESES = 12
JANELA_TESTE_MESES = 6
PASSO_MESES = 6

MINIMO_DADOS = 0.95
TOP_N_LIQUIDEZ = 100
MINIMO_NEGOCIOS_MEDIO = 10

### 2. Função para selecionar ativos líquidos em uma janela

Nesta etapa, criamos uma função para selecionar os ativos líquidos dentro de uma janela de formação. A função utiliza três critérios principais: disponibilidade de dados, número médio de negócios e volume financeiro médio.

Primeiro, mantemos apenas ativos com preços disponíveis em pelo menos 95% dos pregões da janela. Depois, filtramos ativos com negociação média mínima. Por fim, ranqueamos os ativos restantes pelo volume financeiro médio e selecionamos os 100 mais líquidos.

Essa função será aplicada repetidamente nas janelas móveis do projeto.

In [3]:
def selecionar_ativos_liquidos(precos, volumes, negocios, inicio, fim):
    precos_janela = precos.loc[inicio:fim]
    volumes_janela = volumes.loc[inicio:fim]
    negocios_janela = negocios.loc[inicio:fim]

    percentual_dados = precos_janela.notna().mean()

    ativos_validos = percentual_dados[percentual_dados >= MINIMO_DADOS].index

    precos_janela = precos_janela[ativos_validos]
    volumes_janela = volumes_janela[ativos_validos]
    negocios_janela = negocios_janela[ativos_validos]

    negocios_medios = negocios_janela.mean()

    ativos_validos = negocios_medios[negocios_medios >= MINIMO_NEGOCIOS_MEDIO].index

    volumes_janela = volumes_janela[ativos_validos]

    volume_medio = volumes_janela.mean().sort_values(ascending=False)

    ativos_liquidos = volume_medio.head(TOP_N_LIQUIDEZ).index.tolist()

    return ativos_liquidos

### 3. Teste da seleção de ativos líquidos em uma janela

Após criar a função de seleção, testamos sua aplicação em uma janela inicial de formação. Esse teste permite verificar se os filtros estão funcionando corretamente.

In [4]:
ativos_2010 = selecionar_ativos_liquidos(
    precos=precos,
    volumes=volumes_financeiros,
    negocios=negocios,
    inicio="2010-01-01",
    fim="2010-12-31"
)

print(f"Quantidade de ativos selecionados: {len(ativos_2010)}")
ativos_2010[:20]

Quantidade de ativos selecionados: 100


['VALE5',
 'PETR4',
 'OGXP3',
 'ITUB4',
 'BVMF3',
 'PETR3',
 'VALE3',
 'BBDC4',
 'USIM5',
 'GGBR4',
 'BBAS3',
 'CSNA3',
 'PDGR3',
 'ITSA4',
 'CYRE3',
 'AMBV4',
 'CIEL3',
 'FIBR3',
 'MILK11',
 'BRFS3']

### 4. Geração das janelas móveis

Após testar a seleção de ativos líquidos em uma janela específica, o próximo passo é repetir esse processo ao longo de todo o período histórico. Para isso, criamos janelas móveis com 12 meses de formação e 6 meses de teste.

A janela de formação será usada para selecionar os ativos líquidos e, posteriormente, formar os pares. A janela de teste será usada para avaliar a estratégia fora da amostra. Esse procedimento evita o uso de informações futuras e deixa a análise mais rigorosa.

In [5]:
janelas = []

data_inicio = precos.index.min()
data_final = precos.index.max()

inicio_formacao = data_inicio

while True:
    fim_formacao = inicio_formacao + pd.DateOffset(months=JANELA_FORMACAO_MESES) - pd.DateOffset(days=1)
    inicio_teste = fim_formacao + pd.DateOffset(days=1)
    fim_teste = inicio_teste + pd.DateOffset(months=JANELA_TESTE_MESES) - pd.DateOffset(days=1)

    if fim_teste > data_final:
        break

    janelas.append({
        "inicio_formacao": inicio_formacao,
        "fim_formacao": fim_formacao,
        "inicio_teste": inicio_teste,
        "fim_teste": fim_teste
    })

    inicio_formacao = inicio_formacao + pd.DateOffset(months=PASSO_MESES)

janelas = pd.DataFrame(janelas)

janelas.head()

,inicio_formacao,fim_formacao,inicio_teste,fim_teste
0,2010-01-04,2011-01-03,2011-01-04,2011-07-03
1,2010-07-04,2011-07-03,2011-07-04,2012-01-03
2,2011-01-04,2012-01-03,2012-01-04,2012-07-03
3,2011-07-04,2012-07-03,2012-07-04,2013-01-03
4,2012-01-04,2013-01-03,2013-01-04,2013-07-03


### 5. Seleção dos ativos líquidos em todas as janelas móveis

Nesta etapa, aplicamos a função de seleção de ativos líquidos em todas as janelas móveis criadas anteriormente.

O resultado será uma tabela em que cada linha representa um ativo selecionado em uma determinada janela de formação. Essa base será o universo dinâmico de ações líquidas usado nas próximas etapas do projeto.

In [6]:
universo_liquido = []

for _, janela in janelas.iterrows():
    ativos = selecionar_ativos_liquidos(
        precos=precos,
        volumes=volumes_financeiros,
        negocios=negocios,
        inicio=janela["inicio_formacao"],
        fim=janela["fim_formacao"]
    )

    for ativo in ativos:
        universo_liquido.append({
            "inicio_formacao": janela["inicio_formacao"],
            "fim_formacao": janela["fim_formacao"],
            "inicio_teste": janela["inicio_teste"],
            "fim_teste": janela["fim_teste"],
            "ativo": ativo
        })

universo_liquido = pd.DataFrame(universo_liquido)

universo_liquido.head()

,inicio_formacao,fim_formacao,inicio_teste,fim_teste,ativo
0,2010-01-04,2011-01-03,2011-01-04,2011-07-03,VALE5
1,2010-01-04,2011-01-03,2011-01-04,2011-07-03,PETR4
2,2010-01-04,2011-01-03,2011-01-04,2011-07-03,OGXP3
3,2010-01-04,2011-01-03,2011-01-04,2011-07-03,ITUB4
4,2010-01-04,2011-01-03,2011-01-04,2011-07-03,BVMF3


In [7]:
print(f"Quantidade de linhas: {universo_liquido.shape[0]}")
print(f"Quantidade de janelas: {universo_liquido[['inicio_formacao', 'fim_formacao']].drop_duplicates().shape[0]}")
print(f"Quantidade de ativos únicos: {universo_liquido['ativo'].nunique()}")

universo_liquido.groupby(["inicio_formacao", "fim_formacao"])["ativo"].count().head()

Quantidade de linhas: 2900
Quantidade de janelas: 29
Quantidade de ativos únicos: 248


inicio_formacao  fim_formacao
2010-01-04       2011-01-03      100
2010-07-04       2011-07-03      100
2011-01-04       2012-01-03      100
2011-07-04       2012-07-03      100
2012-01-04       2013-01-03      100
Name: ativo, dtype: int64

### 6. Salvamento do universo líquido por janela

Após construir o universo líquido em todas as janelas móveis, salvamos a tabela final em CSV. Essa base será usada nas próximas etapas.

In [8]:
universo_liquido.to_csv(
    PASTA_DADOS_TRATADOS / "universo_liquido_por_janela.csv",
    index=False
)

print("Universo líquido salvo com sucesso.")

Universo líquido salvo com sucesso.
